# 02 - Tools and MCP

**Case study:** Maple quotes an indicative rate, pre-qualifies a borrower, then escalates a self-employed edge case, with a compliance approval gate.

## What you'll learn
- **Agent Framework:** function tools with `tool(...)`, seeing every tool call, `approval_mode="always_require"`, remote and local (stdio) MCP
- **Foundry registration (`AIProjectClient`):** a function-tool *schema* on a prompt agent, a server-side `MCPTool`, a project **connection**, and a **toolbox**; list/get each, then delete
- Which tool pieces can *not* be registered (function bodies, local MCP)

## 1. Setup

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `shared` importable (notebook cwd = notebooks/)

# Boilerplate only: env loading, the Maple system prompt, mock mortgage data, PII redaction.
from shared.fsi_common import load_settings, print_env_check, MAPLE_INSTRUCTIONS, redact

settings = load_settings(require=("project_endpoint", "model"))
print_env_check(settings)

from azure.identity import DefaultAzureCredential
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
client = FoundryChatClient(project_endpoint=settings.project_endpoint, model=settings.model, credential=credential)
pc = AIProjectClient(endpoint=settings.project_endpoint, credential=credential, allow_preview=True)

def show_calls(resp):
    """Tool-call visibility: print every call and result in a response."""
    for m in resp.messages:
        for c in m.contents:
            if c.type in ("function_call", "mcp_server_tool_call"):
                print(f"  CALL   {getattr(c, 'tool_name', None) or c.name}({redact(str(c.arguments))})")
            elif c.type in ("function_result", "mcp_server_tool_result"):
                print(f"  RESULT {redact(str(c.result))[:160]}")

AZURE_SUBSCRIPTION_ID                      set
AZURE_RESOURCE_GROUP                       set
AZURE_LOCATION                             set
AZURE_AI_PROJECT_ENDPOINT                  set
AZURE_AI_MODEL_DEPLOYMENT_NAME             set
AZURE_AI_EMBEDDING_DEPLOYMENT_NAME         set
AZURE_SEARCH_ENDPOINT                      set
AZURE_SEARCH_INDEX_NAME                    set
AZURE_SEARCH_KNOWLEDGE_BASE_NAME           set
AZURE_AI_KB_CONNECTION_NAME                set
APPLICATIONINSIGHTS_CONNECTION_STRING      set
AZURE_CONTAINER_REGISTRY_NAME              set


## 2. Function tools (agent-side)
The five mortgage functions are plain Python in `shared/fsi_common.py` (mock, fictional, deterministic data: boilerplate). **We wrap them as tools here**: `tool(fn, approval_mode=...)`. The model reads each function's name, type hints and docstring to decide when to call it.

In [2]:
import inspect
from shared.fsi_common import get_rate_quote, prequalify, get_document_checklist, lookup_customer, escalate_to_underwriter, audit_log

NEVER = {"approval_mode": "never_require"}
read_only_tools = [tool(get_rate_quote, **NEVER), tool(prequalify, **NEVER), tool(get_document_checklist, **NEVER), tool(lookup_customer, **NEVER)]
for fn in (get_rate_quote, prequalify, get_document_checklist):
    print(fn.__name__, inspect.signature(fn))

get_rate_quote (term_years: 'int', credit_band: 'str', ltv_pct: 'float') -> 'dict'
prequalify (annual_income: 'float', monthly_debts: 'float', down_payment: 'float', price: 'float') -> 'dict'
get_document_checklist (employment_type: 'str') -> 'list[str]'


In [3]:
agent = Agent(client, MAPLE_INSTRUCTIONS, name="maple-tools", tools=read_only_tools)
q = ("Borrower: 120000 income, 600 monthly debts, buying a 500000 home with 100000 down, good credit. "
     "Pre-qualify them and quote an indicative 5-year rate.")
resp = await agent.run(q)
show_calls(resp)
print(redact(resp.text))

  CALL   prequalify({"annual_income":120000,"down_payment":100000,"monthly_debts":600,"price":500000})
  CALL   get_rate_quote({"credit_band":"good","ltv_pct":80,"term_years":5})
  RESULT {"gds_pct": 30.8, "tds_pct": 36.8, "ltv_pct": 80.0, "decision": "likely", "reasons": ["Within guideline limits"], "disclosure": "Indicative only. Not a commitme
  RESULT {"rate_pct": 4.79, "apr_pct": 4.86, "term_years": 5, "as_of": "2026-11-01", "indicative": true, "disclosure": "Indicative only. Not a commitment to lend. Rates 
The borrower is likely pre-qualified for the home purchase with an 80% LTV, a GDS of about 30.8%, and a TDS of about 36.8%, all within guideline limits.

An indicative 5-year fixed rate quote for good credit and 80% LTV is 4.79% (APR 4.86%).

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.


## 3. Approval gate on a sensitive tool
Escalating to an underwriter creates an audit record. Mark it `approval_mode="always_require"`: the run pauses and returns an approval *request*; a human approves or denies, and we resume in the same session.

In [4]:
from agent_framework import Message

gated_tools = read_only_tools + [tool(escalate_to_underwriter, approval_mode="always_require")]
gated = Agent(client, MAPLE_INSTRUCTIONS, name="maple-gated", tools=gated_tools)
session = gated.create_session()
q = "Case C-1002 is self-employed with variable income and a large gift. Please escalate to the underwriter."
resp = await gated.run(q, session=session)
for req in resp.user_input_requests:
    print("APPROVAL REQUESTED:", req.function_call.name, redact(str(req.function_call.arguments)))
print("audit log before approval:", audit_log())

APPROVAL REQUESTED: escalate_to_underwriter {"case_id":"C-1002","reason":"Self-employed with variable income and a large gift, requires human underwriting review."}
audit log before approval: []


In [5]:
APPROVE = True   # change to False to see a denial
msgs = [q, Message("assistant", resp.user_input_requests)]
msgs += [Message("user", [r.to_function_approval_response(APPROVE)]) for r in resp.user_input_requests]
resp2 = await gated.run(msgs, session=session)
print(redact(resp2.text))
print("audit log after:", audit_log())

Case C-1002 has been escalated to a human underwriter for review due to self-employment with variable income and a large gift. If you need anything else, just let me know!
audit log after: [{'ticket_id': 'UW-09427', 'case_id': 'C-1002', 'reason': 'Self-employed with variable income and a large gift, requires human underwriting review.'}]


## 4. Remote MCP: Microsoft Learn (agent-side)
[MCP](https://modelcontextprotocol.io) standardizes how agents discover and call tools. `MCPStreamableHTTPTool` connects *from your process* to a remote server; tools are discovered at runtime.

In [6]:
from agent_framework import MCPStreamableHTTPTool

LEARN_URL = "https://learn.microsoft.com/api/mcp"
async with MCPStreamableHTTPTool("learn", LEARN_URL) as learn:
    print("discovered tools:", [t.name for t in learn.functions])
    a = Agent(client, "Answer briefly, using the Learn tools for Azure facts. Cite the URL.", name="maple-learn", tools=[learn])
    r = await a.run("According to Microsoft Learn, what is a Foundry hosted agent? Two sentences.")
    show_calls(r)
    print(redact(r.text))

discovered tools: ['microsoft_docs_search', 'microsoft_code_sample_search', 'microsoft_docs_fetch']


  CALL   microsoft_docs_search({"query":"Foundry hosted agent"})
  RESULT {"results": [{"title": "What are hosted agents?", "content": "# What are hosted agents?\nWhen you build agentic applications by using open-source frameworks, yo
A Foundry hosted agent is a containerized agent application deployed to Microsoft-managed infrastructure via the Microsoft Foundry Agent Service, where the platform handles scaling, session state persistence, security, and lifecycle management, allowing you to focus on your agent's logic. It supports custom agent code or preferred frameworks, provides a dedicated endpoint and identity for secure access, and manages session persistence and scaling automatically. 

Source: https://learn.microsoft.com/azure/foundry/agents/concepts/hosted-agents


## 5. Local MCP server (stdio) - *not registerable*
`shared/mcp_loan_server.py` is a ~20-line MCP server exposing two Northwind tools. The agent launches it as a **local subprocess** and discovers the tools. Because it runs on your machine, Foundry cannot reach it: **a local stdio MCP server cannot be registered with Foundry**. It is wired purely in code, as below. (To share it, host it at an HTTPS URL and use section 7.)

In [7]:
print(open(pathlib.Path.cwd().parent / "shared" / "mcp_loan_server.py").read())

"""Tiny local MCP server (stdio) exposing Maple mortgage helpers. Run: python mcp_loan_server.py"""
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1]))
from mcp.server.mcpserver import MCPServer  # noqa: E402

from shared import fsi_common as f  # noqa: E402

mcp = MCPServer("northwind-loan-tools")


@mcp.tool()
def document_checklist(employment_type: str) -> list[str]:
    """Documents required for a mortgage application (salaried, self-employed, retired)."""
    return f.get_document_checklist(employment_type)


@mcp.tool()
def indicative_rate(term_years: int, credit_band: str, ltv_pct: float) -> dict:
    """Indicative (non-binding) fixed rate quote."""
    return f.get_rate_quote(term_years, credit_band, ltv_pct)


if __name__ == "__main__":
    mcp.run(transport="stdio")



In [8]:
import os
from agent_framework import MCPStdioTool
from mcp.client.stdio import StdioServerParameters, stdio_client

server = pathlib.Path.cwd().parent / "shared" / "mcp_loan_server.py"

class NotebookStdioTool(MCPStdioTool):
    """Jupyter's stderr has no fileno(), which the stdio client needs: send server stderr to a real file."""
    def get_mcp_client(self):
        params = StdioServerParameters(command=self.command, args=self.args, env=self.env)
        return stdio_client(params, errlog=open(os.devnull, "w"))

async with NotebookStdioTool("northwind-loan", sys.executable, args=[str(server)]) as loan:
    print("discovered tools:", [t.name for t in loan.functions])
    a = Agent(client, MAPLE_INSTRUCTIONS, name="maple-mcp", tools=[loan])
    r = await a.run("What documents does a self-employed applicant need to provide?")
    show_calls(r)
    print(redact(r.text))

discovered tools: ['document_checklist', 'indicative_rate']


  CALL   document_checklist({"employment_type":"self-employed"})
  RESULT {"result": ["Government photo ID", "Two years' Notices of Assessment and T1 Generals", "Business registration or articles of incorporation", "Two years' financi
A self-employed applicant typically needs to provide:
- Government photo ID
- Two years' Notices of Assessment and T1 Generals
- Business registration or articles of incorporation
- Two years' financial statements
- 90 days of business and personal bank statements
- Down-payment source confirmation

If you need details for other employment types or have more questions, just ask!


## 6. Part B - tools registered with Foundry
Everything above ran in your process. Now the same ideas as **server-side assets** in the project. We track what we create in `created` so the last section can delete it.

In [9]:
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, MCPTool

created = {"agents": [], "toolboxes": [], "connections": []}
print("project tools already registered ->",
      "toolboxes:", [t.name for t in pc.toolboxes.list()], "| connections:", [c.name for c in pc.connections.list()])

project tools already registered -> toolboxes: [] | connections: ['appinsights', 'search', 'northwind-kb-mcp', 'maple-kb-mcp-nb03']


### 6.1 Function tool schema on a prompt agent
A prompt agent stores the *schema* of your function (`FunctionTool`: name, description, JSON schema). The **body stays in your code**: when the model asks for the call, the Responses API returns a `function_call` item and *you* run the function and send the output back. That is the (c) case: Foundry registers the contract, not the code.

In [10]:
rate_schema = FunctionTool(
    name="get_rate_quote",
    description="Indicative fixed-rate quote from the Northwind rate sheet. Not a commitment.",
    parameters={"type": "object", "additionalProperties": False,
                "properties": {"term_years": {"type": "integer"}, "credit_band": {"type": "string", "enum": ["excellent", "good", "fair", "poor"]},
                               "ltv_pct": {"type": "number"}},
                "required": ["term_years", "credit_band", "ltv_pct"]},
    strict=True,
)
FN_AGENT = "nb02-maple-fn"
v = pc.agents.create_version(agent_name=FN_AGENT, definition=PromptAgentDefinition(
        model=settings.model, instructions=MAPLE_INSTRUCTIONS, tools=[rate_schema]),
        description="Maple with get_rate_quote function schema (nb02)")
created["agents"].append(FN_AGENT)
print("registered", v.name, "v", v.version, "| tools stored:", [t["type"] + ":" + t.get("name", "") for t in pc.agents.get_version(FN_AGENT, v.version)["definition"]["tools"]])

registered nb02-maple-fn v 1 | tools stored: ['function:get_rate_quote']


In [11]:
import json
oc = pc.get_openai_client(agent_name=FN_AGENT)
r = oc.responses.create(input="Indicative 5-year fixed rate, good credit, 75% LTV?")
calls = [o for o in r.output if o.type == "function_call"]
for c in calls:
    print("model asked Foundry-side agent to call:", c.name, c.arguments)
outputs = [{"type": "function_call_output", "call_id": c.call_id, "output": json.dumps(get_rate_quote(**json.loads(c.arguments)))} for c in calls]   # OUR code runs the body
r2 = oc.responses.create(previous_response_id=r.id, input=outputs)
print(redact(r2.output_text))

model asked Foundry-side agent to call: get_rate_quote {"term_years":5,"credit_band":"good","ltv_pct":75}


The indicative 5-year fixed rate for a borrower with good credit and a 75% LTV is 4.79% (APR 4.86%). Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.


### 6.2 Server-side MCP tool on a prompt agent
With a prompt agent, **Foundry** calls the MCP server for you (no client-side loop): add an `MCPTool` to the definition. The server must be reachable from Foundry (public HTTPS, or via a project connection).

In [12]:
MCP_AGENT = "nb02-maple-learn-mcp"
v = pc.agents.create_version(agent_name=MCP_AGENT, definition=PromptAgentDefinition(
        model=settings.model, instructions="Answer briefly using Microsoft Learn. Cite the URL.",
        tools=[MCPTool(server_label="learn", server_url=LEARN_URL, require_approval="never")]))
created["agents"].append(MCP_AGENT)
oc = pc.get_openai_client(agent_name=MCP_AGENT)
r = oc.responses.create(input="Using Learn, what is a Foundry hosted agent? One sentence.")
print("output item types:", [o.type for o in r.output])
print(redact(r.output_text))

output item types: ['mcp_list_tools', 'mcp_call', 'message']
A Foundry hosted agent is a containerized agent application deployed on Microsoft-managed infrastructure through Foundry Agent Service, which handles scaling, session state persistence, security, and lifecycle management, allowing you to focus on your agent's logic and orchestration while securely operating AI agents at scale. 

Source: https://learn.microsoft.com/azure/foundry/agents/concepts/hosted-agents


### 6.3 A project connection
A **connection** stores a target URL plus how to authenticate to it (managed identity, key, none) so agents reference it by name (`project_connection_id`) and secrets never sit in agent definitions. The Python data-plane client can **list and get** connections but not create them; connections are ARM resources, so we create one with a REST call (the same API Bicep uses).

In [13]:
import requests
from urllib.parse import urlparse

host, path = urlparse(settings.project_endpoint), urlparse(settings.project_endpoint).path
account, project = host.hostname.split(".")[0], path.rstrip("/").split("/")[-1]
CONN = "learn-mcp-conn"
arm = (f"https://management.azure.com/subscriptions/{settings.subscription_id}/resourceGroups/{settings.resource_group}"
       f"/providers/Microsoft.CognitiveServices/accounts/{account}/projects/{project}/connections/{CONN}?api-version=2025-10-01-preview")
tok = credential.get_token("https://management.azure.com/.default").token
resp = requests.put(arm, headers={"Authorization": f"Bearer {tok}"},
                    json={"properties": {"category": "RemoteTool", "target": LEARN_URL, "authType": "None", "isSharedToAll": False,
                                         "metadata": {"type": "custom_MCP"}}})
print("ARM PUT ->", resp.status_code, resp.json().get("properties", {}).get("target") if resp.ok else resp.text[:300])
created["connections"].append(CONN)
c = pc.connections.get(CONN)
print("get ->", c.name, c.type, c.target)
print("all connections:", [x.name for x in pc.connections.list()])

ARM PUT -> 200 https://learn.microsoft.com/api/mcp
get -> learn-mcp-conn RemoteTool https://learn.microsoft.com/api/mcp


all connections: ['appinsights', 'search', 'northwind-kb-mcp', 'learn-mcp-conn']


In [14]:
# an agent can now reference the connection by name instead of embedding details
CONN_AGENT = "nb02-maple-mcp-conn"
v = pc.agents.create_version(agent_name=CONN_AGENT, definition=PromptAgentDefinition(
        model=settings.model, instructions="Answer briefly using Microsoft Learn. Cite the URL.",
        tools=[MCPTool(server_label="learn", server_url=LEARN_URL, require_approval="never", project_connection_id=CONN)]))
created["agents"].append(CONN_AGENT)
r = pc.get_openai_client(agent_name=CONN_AGENT).responses.create(input="Using Learn: what is Microsoft Foundry? One sentence.")
print(redact(r.output_text))

Microsoft Foundry unifies agents, models, and tools under a single management grouping with built-in enterprise-readiness capabilities including tracing, monitoring, evaluations, and customizable enterprise setup configurations, providing streamlined management through unified role-based access control (RBAC), networking, and policies under one Azure resource provider namespace.  
Reference: https://learn.microsoft.com/azure/foundry/what-is-foundry


### 6.4 A toolbox
A **toolbox** is a named, versioned bundle of tools (MCP servers, Azure AI Search, code interpreter, OpenAPI...) governed centrally and exposed through **one MCP endpoint**, so many agents share one approved tool set. Portal: **Foundry project -> Build -> Tools -> Toolboxes**.

In [15]:
from azure.ai.projects.models import MCPToolboxTool

TB = "nb02-maple-toolbox"
tv = pc.toolboxes.create_version(TB, tools=[MCPToolboxTool(server_label="learn", server_url=LEARN_URL, description="Microsoft Learn docs")],
                                 description="Maple approved tools (nb02)")
created["toolboxes"].append(TB)
print("created toolbox", tv.name, "version", tv.version)
print("list ->", [t.name for t in pc.toolboxes.list()])
tb = pc.toolboxes.get(TB)
print("get  -> default version:", tb.default_version if hasattr(tb, "default_version") else tb)
print("versions ->", [v.version for v in pc.toolboxes.list_versions(TB)])

created toolbox

 nb02-maple-toolbox version 1


list -> ['nb02-maple-toolbox']
get  -> default version: 1


versions -> ['1']


### 6.5 Built-in catalog tools
Foundry also ships built-in tools (code interpreter, web search, file search, Azure AI Search...). They need no registration: you add them to an agent. Code interpreter here does the amortization maths with real code (AF-side helper on the chat client).

In [16]:
calc = Agent(client, "You are a careful mortgage calculator. Use code for arithmetic.", name="maple-calc", tools=[client.get_code_interpreter_tool()])
try:
    r = await calc.run("Monthly payment on a 400000 loan at 4.79% over 25 years, monthly compounding? Give one number.")
    print(redact(r.text))
except Exception as e:
    print("Code interpreter unavailable here:", type(e).__name__, str(e)[:300])

To calculate the monthly payment on a loan, we use the formula for an amortizing loan:

\[
M = P \frac{r(1+r)^n}{(1+r)^n - 1}
\]

Where:
- \(M\) = monthly payment
- \(P\) = principal loan amount = 400,000
- \(r\) = monthly interest rate = annual rate / 12 = 4.79% / 12
- \(n\) = number of payments = years × 12 = 25 × 12

Let's calculate it.The monthly payment on a $400,000 loan at 4.79% interest over 25 years with monthly compounding is approximately $2289.68.


## 7. What is *not* registerable (summary)
| Piece | Registered? | How it is wired |
|---|---|---|
| `@tool` function **body** | No | runs in your process (AF), or your loop handles `function_call` items (6.1); Foundry stores only the schema |
| Local stdio MCP server | No | `MCPStdioTool` launches a subprocess (section 5) |
| Remote MCP server | Yes (as `MCPTool` on an agent / toolbox / connection) | 6.2-6.4 |
| Approval gate on function tools | No (client-side policy) | `approval_mode` (section 3); for server-side MCP use `require_approval` |

## 8. Clean up what this notebook registered
Deleting an agent removes all its versions; toolboxes and connections have their own delete calls.

In [17]:
for a in created["agents"]:
    pc.agents.delete(agent_name=a)
for t in created["toolboxes"]:
    pc.toolboxes.delete(t)
for c in created["connections"]:
    rr = requests.delete(arm.replace(CONN, c), headers={"Authorization": f"Bearer {credential.get_token('https://management.azure.com/.default').token}"})
    print("delete connection", c, "->", rr.status_code)
print("agents:", sorted(a.name for a in pc.agents.list()))
print("toolboxes:", [t.name for t in pc.toolboxes.list()], "| connections:", [c.name for c in pc.connections.list()])
assert not set(created["agents"]) & {a.name for a in pc.agents.list()}

delete connection learn-mcp-conn -> 200


agents: ['maple-hosted', 'maple-learn-mcp', 'maple-prompt']


toolboxes: [] | connections: ['appinsights', 'search', 'northwind-kb-mcp']


## Key takeaways
- Plain Python functions become tools with `tool(...)`; type hints and docstrings are the contract. Inspect tool calls: they are your audit trail.
- `always_require` gives a human-in-the-loop gate; resume in the same session.
- Foundry registers *descriptions and endpoints* (function schema, `MCPTool`, connections, toolboxes), never your function bodies or local servers.
- Connections keep auth out of agent definitions; toolboxes centralize governed tool access for many agents.

**Next up:** [03-foundry-iq-knowledge.ipynb](03-foundry-iq-knowledge.ipynb) - ground the agent in underwriting policy.